# Study 0 end-to-end check on a fresh development split, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding. Text only. No compressor.

Uses only `data/plan_fresh.jsonl` (6 families, 18 examples). Do **not** load `data/plan_dev.jsonl`, `data/test.jsonl`, `hop_test`, or a `plan_test` file.

Do **not** resume `results_stage_plan0`, `results_stage_plan0_corrected`, or `results_stage_plan0_diag`. Keep those runs.

Frozen prompts from the diagnostic. No further prompt tuning.

- A generates the compact eligibility message.
- B uses reason-then-answer on that generated message (`collab_reason`).
- Control: B uses reason-then-answer on a mismatched A message (`mismatch_reason`).
- A's three labels are scored (`a_status`). This is not an oracle-to-B run.

18 A messages + 36 B answers. Resume is on for `/kaggle/working/results_stage_plan0_e2e` only. Download that folder during the run.

Keep every example. These are 6 families with three versions each, not 18 independent scenarios, and not held-out Study 1.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan0_e2e.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan0_e2e.py").is_file() and (cand / "data" / "plan_fresh.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan0_e2e.py and data/plan_fresh.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan0_e2e.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan_fresh", sum(1 for _ in (REPO / "data" / "plan_fresh.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan_test.jsonl").exists()

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan0_e2e")
print("Do not resume results_stage_plan0, results_stage_plan0_corrected, or results_stage_plan0_diag")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan0_e2e", "--split", "plan_fresh"])
print("plan0 e2e exit", rc)

In [ ]:
src = WORK / "results_stage_plan0_e2e"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan0_e2e" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:8000])